# Partie 2 — Traitement batch avec SparkCore (RDD)

**Prérequis** (voir `README.md`) : le fichier `orders_history.jsonl` (50 000 événements) a été généré
avec `producer.py --mode file --n 50000` puis copié dans `lake/bronze/history/`, et
`restaurants.json` dans `lake/bronze/reference/`.

> Laisser la session ouverte pendant les captures : la Spark UI (DAG, stages) est sur
> http://localhost:4040 tant que `spark.stop()` n'a pas été exécuté.

In [1]:
import json, time, os
from operator import add
import pandas as pd
from spark_session import get_spark, LAKE

spark = get_spark("FastFood-Partie2-RDD")
sc = spark.sparkContext

HISTORY = f"{LAKE}/bronze/history/orders_history.jsonl"
RESTAURANTS = f"{LAKE}/bronze/reference/restaurants.json"
RESULTS = "/home/jovyan/work/results"
os.makedirs(RESULTS, exist_ok=True)

timings = {}
def chrono(label, action):
    t0 = time.perf_counter()
    res = action()
    timings[label] = round(time.perf_counter() - t0, 3)
    print(f"[temps] {label} : {timings[label]} s")
    return res

## 1. Chargement en RDD (`sc.textFile`) et parsing JSON

`textFile`, `flatMap` sont des **transformations** : rien n'est lu tant qu'aucune **action**
(`count`, `take`, `collect`...) n'est appelée. Les lignes mal formées sont ignorées
(`flatMap` renvoie une liste vide).

In [2]:
raw = sc.textFile(HISTORY, minPartitions=4)          # transformation (lazy)

def parse(line):
    try:
        return [json.loads(line)]
    except ValueError:
        return []                                     # ligne invalide -> ignorée

orders = raw.flatMap(parse)                           # transformation (lazy)
orders.cache()                                        # réutilisé plusieurs fois

n_events = chrono("count (lecture MinIO + parsing)", orders.count)   # ACTION
print("Partitions :", orders.getNumPartitions(), "| événements :", n_events)
orders.take(2)                                                        # ACTION

[temps] count (lecture MinIO + parsing) : 73.372 s
Partitions : 4 | événements : 50000


[{'order_id': 'ORD-0000001',
  'restaurant_id': 'R-168',
  'customer_id': 'C-11420',
  'city': 'Tanger',
  'amount_mad': 192.32,
  'items_count': 2,
  'distance_km': 2.9,
  'courier_id': 'L-TAN-05',
  'status': 'CREATED',
  'event_time': '2026-10-16T00:18:01Z'},
 {'order_id': 'ORD-0000001',
  'restaurant_id': 'R-168',
  'customer_id': 'C-11420',
  'city': 'Tanger',
  'amount_mad': 192.32,
  'items_count': 2,
  'distance_km': 2.9,
  'courier_id': 'L-TAN-05',
  'status': 'PICKED_UP',
  'event_time': '2026-10-16T00:31:34Z'}]

In [3]:
# Répartition des événements par statut
sorted(orders.map(lambda o: (o["status"], 1)).reduceByKey(add).collect())

[('CANCELLED', 1347),
 ('CREATED', 17007),
 ('DELIVERED', 15635),
 ('PICKED_UP', 16011)]

## 2. `filter` : commandes `DELIVERED` uniquement

Le générateur produit ~2 % de **doublons** (même `order_id` + même `status`). Sans dédoublonnage,
le chiffre d'affaires serait surestimé : on garde une seule ligne par `order_id`.

In [4]:
delivered = orders.filter(lambda o: o["status"] == "DELIVERED")                 # transformation

delivered_unique = (delivered
                    .map(lambda o: (o["order_id"], o))
                    .reduceByKey(lambda a, b: a)                                 # 1 ligne par commande
                    .values())
delivered_unique.cache()

n_deliv = delivered.count()
n_uniq = delivered_unique.count()
print(f"DELIVERED : {n_deliv} lignes, {n_uniq} commandes distinctes -> {n_deliv - n_uniq} doublons supprimés")

DELIVERED : 15635 lignes, 15313 commandes distinctes -> 322 doublons supprimés


## 3. `map` + `reduceByKey` : chiffre d'affaires total par ville

In [5]:
ca_ville = (delivered_unique
            .map(lambda o: (o["city"], o["amount_mad"]))
            .reduceByKey(add))                                    # transformation -> shuffle

sc.setJobGroup("ca_ville", "CA par ville (reduceByKey)")
ca = chrono("CA par ville (RDD, données en cache)", ca_ville.collect)   # ACTION

tracker = sc.statusTracker()
for job_id in tracker.getJobIdsForGroup("ca_ville"):
    print(f"job {job_id} -> stages {sorted(tracker.getJobInfo(job_id).stageIds)}")
sc.setLocalProperty("spark.jobGroup.id", None)          # fin du groupe : les jobs suivants n'y sont plus rattachés
sc.setLocalProperty("spark.job.description", None)

pd.DataFrame(sorted(ca, key=lambda kv: -kv[1]), columns=["ville", "CA (MAD)"]).round(2)

[temps] CA par ville (RDD, données en cache) : 4.28 s
job 5 -> stages [7, 8, 9]


,ville,CA (MAD)
0,Casablanca,1079464.82
1,Rabat,596828.52
2,Marrakech,503885.09
3,Tanger,410253.37


## 4. Panier moyen par ville — somme **et** nombre en un seul passage (`aggregateByKey`)

- `zeroValue = (0.0, 0)` : accumulateur (somme, nombre) ;
- `seqOp` : ajoute une commande à l'accumulateur **dans** une partition ;
- `combOp` : fusionne les accumulateurs de deux partitions (après le shuffle).

In [6]:
zero = (0.0, 0)
seq_op = lambda acc, montant: (acc[0] + montant, acc[1] + 1)
comb_op = lambda a, b: (a[0] + b[0], a[1] + b[1])

stats_ville = (delivered_unique
               .map(lambda o: (o["city"], o["amount_mad"]))
               .aggregateByKey(zero, seq_op, comb_op))

panier = stats_ville.mapValues(lambda s: (s[0], s[1], s[0] / s[1]))
res_panier = chrono("panier moyen (aggregateByKey)", panier.collect)          # ACTION

df_ville = (pd.DataFrame([(v, ca_, n, pm) for v, (ca_, n, pm) in res_panier],
                         columns=["ville", "CA (MAD)", "commandes livrées", "panier moyen (MAD)"])
            .sort_values("CA (MAD)", ascending=False).round(2).reset_index(drop=True))
df_ville

[temps] panier moyen (aggregateByKey) : 2.472 s


,ville,CA (MAD),commandes livrées,panier moyen (MAD)
0,Casablanca,1079464.82,6874,157.04
1,Rabat,596828.52,3342,178.58
2,Marrakech,503885.09,2834,177.80
3,Tanger,410253.37,2263,181.29


In [7]:
# Variante équivalente avec reduceByKey sur des paires (montant, 1)
alt = (delivered_unique.map(lambda o: (o["city"], (o["amount_mad"], 1)))
       .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
       .mapValues(lambda s: round(s[0] / s[1], 2)))
sorted(alt.collect())

[('Casablanca', 157.04),
 ('Marrakech', 177.8),
 ('Rabat', 178.58),
 ('Tanger', 181.29)]

## 5. Top 5 des restaurants par nombre de commandes livrées (`takeOrdered`)

Les noms viennent du référentiel `restaurants.json`, diffusé à tous les Executors avec une
**variable broadcast** (petite table envoyée une seule fois à chaque Executor).

In [8]:
ref = (sc.textFile(RESTAURANTS)
       .map(json.loads)
       .map(lambda r: (r["restaurant_id"], (r["name"], r["city"], r["category"])))
       .collectAsMap())
ref_bc = sc.broadcast(ref)

par_resto = (delivered_unique
             .map(lambda o: (o["restaurant_id"], 1))
             .reduceByKey(add))

top5 = chrono("top 5 restaurants (takeOrdered)",
              lambda: par_resto.takeOrdered(5, key=lambda kv: -kv[1]))      # ACTION

df_top5 = pd.DataFrame(
    [(rid, *ref_bc.value.get(rid, ("?", "?", "?")), n) for rid, n in top5],
    columns=["restaurant_id", "nom", "ville", "catégorie", "commandes livrées"])
df_top5.index = range(1, 6)
df_top5

[temps] top 5 restaurants (takeOrdered) : 4.561 s


,restaurant_id,nom,ville,catégorie,commandes livrées
1,R-001,Dar Palmier Tacos,Casablanca,tacos,494
2,R-010,Chez Palmier Healthy,Tanger,healthy,373
3,R-002,Bab Corniche Healthy,Rabat,healthy,331
4,R-003,Bab Atlas Tacos,Casablanca,tacos,322
5,R-004,Maison Andalous Marocain,Marrakech,marocain,297


## 6. Lineage (`toDebugString`) — transformations *lazy* vs actions

Chaque niveau d'indentation (`+-`) correspond à une **frontière de shuffle**, donc à un nouveau stage.

In [9]:
print(ca_ville.toDebugString().decode())

(4) PythonRDD[21] at collect at /tmp/ipykernel_3425/1872150068.py:17 []
 |  MapPartitionsRDD[20] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[19] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(4) PairwiseRDD[18] at reduceByKey at /tmp/ipykernel_3425/929797852.py:3 []
    |  PythonRDD[17] at reduceByKey at /tmp/ipykernel_3425/929797852.py:3 []
    |  PythonRDD[14] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 4; MemorySize: 806.9 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[13] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[12] at partitionBy at NativeMethodAccessorImpl.java:0 []
    +-(4) PairwiseRDD[11] at reduceByKey at /tmp/ipykernel_3425/3996074689.py:5 []
       |  PythonRDD[10] at reduceByKey at /tmp/ipykernel_3425/3996074689.py:5 []
       |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
       |      CachedPartitions: 4; MemorySize: 1760.4 KiB; DiskSize: 0.0 B
       |  s3a://lake/bronze/history/orders_history.jsonl MapPartit

In [10]:
print(panier.toDebugString().decode())

(4) PythonRDD[26] at collect at /tmp/ipykernel_3425/1872150068.py:17 []
 |  MapPartitionsRDD[25] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[24] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(4) PairwiseRDD[23] at aggregateByKey at /tmp/ipykernel_3425/1034388443.py:7 []
    |  PythonRDD[22] at aggregateByKey at /tmp/ipykernel_3425/1034388443.py:7 []
    |  PythonRDD[14] at RDD at PythonRDD.scala:53 []
    |      CachedPartitions: 4; MemorySize: 806.9 KiB; DiskSize: 0.0 B
    |  MapPartitionsRDD[13] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[12] at partitionBy at NativeMethodAccessorImpl.java:0 []
    +-(4) PairwiseRDD[11] at reduceByKey at /tmp/ipykernel_3425/3996074689.py:5 []
       |  PythonRDD[10] at reduceByKey at /tmp/ipykernel_3425/3996074689.py:5 []
       |  PythonRDD[2] at RDD at PythonRDD.scala:53 []
       |      CachedPartitions: 4; MemorySize: 1760.4 KiB; DiskSize: 0.0 B
       |  s3a://lake/bronze/history/orders_history.jsonl M

| Opération utilisée | Type | Effet |
|---|---|---|
| `textFile`, `flatMap`, `map`, `filter`, `values`, `mapValues` | Transformation (lazy, *narrow*) | ajoutée au lineage, pas d'exécution |
| `reduceByKey`, `aggregateByKey` | Transformation (lazy, *wide*) | provoque un **shuffle** → nouveau stage |
| `cache` | Marqueur de persistance (lazy) | le RDD sera gardé en mémoire au 1er calcul |
| `count`, `take`, `collect`, `takeOrdered`, `collectAsMap` | **Action** | déclenche un job (DAG → stages → tâches) |

## 7. Mesure de référence pour la Partie 3 (comparaison RDD vs DataFrame)

Calcul complet du CA par ville **sans cache** : lecture MinIO → parsing → filtre → dédoublonnage → agrégation.

In [11]:
def ca_par_ville_rdd():
    return (sc.textFile(HISTORY, minPartitions=4)
              .flatMap(parse)
              .filter(lambda o: o["status"] == "DELIVERED")
              .map(lambda o: (o["order_id"], o)).reduceByKey(lambda a, b: a).values()
              .map(lambda o: (o["city"], o["amount_mad"]))
              .reduceByKey(add)
              .collect())

for i in range(3):
    chrono(f"CA par ville RDD sans cache - essai {i + 1}", ca_par_ville_rdd)

[temps] CA par ville RDD sans cache - essai 1 : 9.483 s


[temps] CA par ville RDD sans cache - essai 2 : 6.828 s


[temps] CA par ville RDD sans cache - essai 3 : 5.9 s


## 8. Sauvegarde des résultats (tableaux du rapport)

In [12]:
df_ville.to_csv(f"{RESULTS}/partie2_ca_panier_par_ville.csv", index=False)
df_top5.to_csv(f"{RESULTS}/partie2_top5_restaurants.csv", index=False)
with open(f"{RESULTS}/partie2_timings.json", "w") as f:
    json.dump(timings, f, indent=2, ensure_ascii=False)
print(json.dumps(timings, indent=2, ensure_ascii=False))

{
  "count (lecture MinIO + parsing)": 73.372,
  "CA par ville (RDD, données en cache)": 4.28,
  "panier moyen (aggregateByKey)": 2.472,
  "top 5 restaurants (takeOrdered)": 4.561,
  "CA par ville RDD sans cache - essai 1": 9.483,
  "CA par ville RDD sans cache - essai 2": 6.828,
  "CA par ville RDD sans cache - essai 3": 5.9
}


## 9. Captures Spark UI pour le rapport

Ouvrir http://localhost:4040 → onglet **Jobs** :

1. Cliquer sur le job du **CA par ville** (description `CA par ville (reduceByKey)`) → **DAG Visualization** :
   on voit **2 stages** séparés par le shuffle de `reduceByKey` (le dédoublonnage ajoute un shuffle,
   mais il est déjà en cache, son stage apparaît en *skipped*).
2. Onglet **Stages** : nombre de tâches par stage (= nombre de partitions), *Shuffle Write* / *Shuffle Read*.
3. Onglet **Storage** : les RDD mis en cache (`orders`, `delivered_unique`).

Une fois les captures faites :

In [13]:
spark.stop()